# Train HYREC emulators

Run this notebook on Google Colab with a GPU runtime.
Before starting: **Runtime → Change runtime type → T4 GPU**.

## Data setup (do once)
Copy the `hyrec-data/` folder to your Google Drive using the
[Google Drive desktop app](https://www.google.com/drive/download/).
The code is cloned directly from GitHub — no need to upload it.

In [ ]:
# Check we have a GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — change runtime type to GPU!')

In [ ]:
# Clone the repo and install
!git clone --branch hyrec-emulator https://github.com/htjb/hyprfine.git
!pip install -q ./hyprfine

In [ ]:
# Install remaining dependencies
!pip install -q "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q astroemu

In [ ]:
import jax
print(jax.devices())  # should show GpuDevice

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Point directly at the synced folder — adjust path if you put it in a subfolder
DATA_DIR = '/content/drive/MyDrive/hyrec-data'
print(f"Files: {len(os.listdir(DATA_DIR))}")

In [ ]:
os.makedirs('/content/emulators', exist_ok=True)
os.makedirs('/content/plots', exist_ok=True)

In [ ]:
import glob
import jax.numpy as jnp
import matplotlib.pyplot as plt
from astroemu.dataloaders import SpectrumDataset
from astroemu.network import mlp
from astroemu.normalisation import log_base_10, standardise
from astroemu.serialisation import load, save
from astroemu.train import train
from astroemu.utils import compute_mean_std

files = glob.glob(f'{DATA_DIR}/*.npz')
print(f'Found {len(files)} files.')
train_files = files[:int(len(files) / 100 * 80)]
val_files   = files[int(len(files) / 100 * 80):int(len(files) / 100 * 90)]
test_files  = files[int(len(files) / 100 * 90):]

In [ ]:
config = {
    'hidden_size': 256,
    'nlayers': 4,
    'act': 'tanh',
    'epochs': 1000,
    'patience': 50,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
}

In [ ]:
ylabels = {'xe': '$x_e$', 'tk': '$T_k$ [K]'}

for label in ['xe', 'tk']:
    print(f'\n--- Training {label} emulator ---')
    variable_input = ['H0', 'omb', 'omc', 'yhe']
    log10 = log_base_10(log_all_y=True, log_all_params=True)

    train_dataset = SpectrumDataset(
        files=train_files, x='z', y=label,
        variable_input=variable_input, tiling=False,
        allow_pickle=True, forward_pipeline=log10,
    )

    _, x, _ = train_dataset[0]

    mean_spec, std_spec, mean_x, std_x, mean_params, std_params = compute_mean_std(
        train_dataset.get_batch_iterator(batch_size=1024, shuffle=False)
    )
    standard = standardise(
        y_mean=mean_spec, y_std=std_spec,
        x_mean=mean_x,   x_std=std_x,
        params_mean=mean_params, params_std=std_params,
    )

    train_dataset.tiling = True
    train_dataset.forward_pipeline = [log10, standard]

    val_dataset = SpectrumDataset(
        files=val_files, x='z', y=label,
        variable_input=variable_input, tiling=True,
        allow_pickle=True, forward_pipeline=[log10, standard],
    )
    test_dataset = SpectrumDataset(
        files=test_files, x='z', y=label,
        variable_input=variable_input, tiling=True,
        allow_pickle=True, forward_pipeline=[log10, standard],
    )

    best_params, train_losses, val_losses = train(
        train_dataset=train_dataset, val_dataset=val_dataset,
        **config, batch_size=512,
    )

    # Training curve
    plt.plot(train_losses, label='Train')
    plt.plot(val_losses, label='Val')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.savefig(f'/content/plots/hyrec_training_curve_{label}.png', dpi=150)
    plt.show()
    plt.close()

    save(
        f'/content/emulators/hyrec_{label}.astroemu',
        best_params, train_losses, val_losses,
        **config, loss='mse',
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
    )

    loaded = load(f'/content/emulators/hyrec_{label}.astroemu')

    predictions, true_values = [], []
    for batch in test_dataset.get_batch_iterator(batch_size=32, shuffle=False):
        y, params = batch
        preds = mlp(loaded['params'], params, act=loaded['hyperparams']['act'])
        preds = preds.reshape(-1, len(x))
        y = y.reshape(-1, len(x))
        for pipe in reversed(test_dataset.forward_pipeline):
            preds, _, _ = pipe.backward(preds, x, params)
            y, _, _ = pipe.backward(y, x, params)
        predictions.append(preds)
        true_values.append(y)

    predictions = jnp.vstack(predictions)
    true_values = jnp.vstack(true_values)

    # Predictions vs truth
    [plt.plot(x, predictions[i], c='r', ls='--') for i in range(10)]
    [plt.plot(x, true_values[i], c='k', ls='-') for i in range(10)]
    plt.loglog()
    plt.xlabel('Redshift $z$')
    plt.ylabel(ylabels[label])
    plt.savefig(f'/content/plots/hyrec_predictions_{label}.png', dpi=150)
    plt.show()
    plt.close()

    # Percentage error
    percent_err = jnp.abs((predictions - true_values) / true_values) * 100
    plt.plot(x, jnp.median(percent_err, axis=0), c='k')
    plt.fill_between(
        x, jnp.percentile(percent_err, 16, axis=0),
        jnp.percentile(percent_err, 84, axis=0),
        alpha=0.3, color='k', label='16th–84th percentile',
    )
    plt.xlabel('Redshift $z$')
    plt.ylabel(f'Percentage error in {ylabels[label]} [%]')
    plt.xscale('log')
    plt.legend()
    plt.savefig(f'/content/plots/hyrec_percent_error_{label}.png', dpi=150)
    plt.show()
    plt.close()

print('\nDone!')

In [ ]:
# Copy outputs back to Google Drive for safekeeping
DRIVE_OUT = '/content/drive/MyDrive/hyprfine-emulators'
!mkdir -p {DRIVE_OUT}
!cp /content/emulators/*.astroemu {DRIVE_OUT}/
!cp /content/plots/*.png {DRIVE_OUT}/
print(f'Saved to {DRIVE_OUT}')